In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-6: GRPO and its refinements

## Where this session fits

> **The limitation we start from: the plain GRPO of Intro session C2 has weak spots, and published recipes show that the details decide whether it works.**

- **The question:** which weak spots of GRPO do later recipes fix, and how?

<img src="images/series_map_6.png" width=80%>

**Position: session 6 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-5 gave PPO in depth, including what its critic (a second large network that estimates values) costs
    - this session is the other branch: GRPO without a critic, and the fixes proposed for it
    - RLLM-7 begins part 2 of the RLLM series, RFT Alternative methods, other ways to do reinforcement fine-tuning (**RFT**, defined in RLLM-1)
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. say what evidence shows that plain GRPO falls short, and what that evidence does not show
2. name each part of the GRPO objective that a later recipe changes, and say why
3. compute a token's **share of the loss** under two averaging rules, and an advantage with and without the standard deviation
    - the share of the loss is how much that token's term counts in the objective
4. say what each change risks, and which changes are arithmetic and which are arguments or design choices

**Intro sessions named in this deck**

- **C1:** "Reinforcement learning on a language model" (the reference model and the KL)
- **C2:** "GRPO: a baseline without a critic" (the objective, in its plain form)
- **B4:** "Policy gradient III: how far may one update move?" (the ratio and the clip)


## Why this session: plain GRPO fell short of a reported result

**Why this section?** Before changing an objective we should see that it needs changing.

> **The DAPO paper reports that a plain GRPO run on a 32-billion-parameter base model scored 30 on AIME 2024; DeepSeek reported 47 for DeepSeek-R1-Zero-Qwen-32B.**

- **AIME 2024:** competition math problems with exact numeric answers
    - the score is averaged over 32 samples per problem
- **DAPO** (Decoupled Clip and Dynamic sAmpling Policy Optimization): an open recipe from ByteDance and collaborators
    - it reaches 50 on the same benchmark, the paper says in about half the training steps
- the gap was closed by a short list of changes, found by diagnosing training runs
- two other papers propose fixes too: **Dr. GRPO** ("Understanding R1-Zero-Like Training") and **GSPO** (Group Sequence Policy Optimization)


## What the DAPO ablation shows, and does not

**Why this section?** To see which changes the evidence supports, one at a time.

> **Each change was added on top of the previous ones; the score rose at every step, from 30 to 50.**

<img src="images/rllm6_ablation.png" width=52%>

- the paper's reported numbers, cumulative in the authors' order: a step's gain depends on the steps before it
- the figure's names are defined later
    - token-level loss: Change 1; dynamic sampling: Change 4
    - overlong filtering, soft overlong punishment: Change 5
- four of the seven changes have a number here (five steps)
    - Dr. GRPO's standard deviation, the KL drop and GSPO's ratio have none
- one base model, one benchmark, the authors' own runs
    - it shows "these changes helped here", not "this is the best recipe", and not a ranking


## The GRPO objective, in words and then in symbols

**Why this section?** To change an objective we must name its parts. Intro session C2 derived it; this is the recall.

> **The objective takes PPO's clipped term per token, multiplies it by one advantage for the whole completion, subtracts a KL term, and averages over tokens, then over the group.**

$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{\lvert \y_i \rvert} \sum_{\tt} \Bigl[ \min{} \bigl( r_{i,\tt}(\theta)\, \hat{\advseq}_i, \;\; \text{clip}\bigl(r_{i,\tt}(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \hat{\advseq}_i \bigr) \; - \; \beta \, \KL_{i,\tt} \Bigr]
\qquad \hat{\advseq}_i = \frac{\rew_i - \text{group mean of } \rew}{\text{group standard deviation of } \rew}
$$

- **ratio** $r_{i,\tt}$: new over old probability of token $\tt$ (B4); the **clip** limits it
- **KL term**, coefficient $\beta$: keeps the policy near the reference model (C1)

**Seven changes, three kinds of reason**

| Part | Claimed weak spot | Change | Source | Kind of reason |
|:--|:--|:--|:--|:--|
| averaging | long completions' tokens count less | all tokens in the group, or a constant | DAPO, Dr. GRPO | arithmetic, shown next |
| advantage scale | extreme questions get more weight | drop the division by std | Dr. GRPO | paper's argument |
| clip | a tight upper clip holds back rare tokens | higher upper clip | DAPO | paper's observation |
| sampling | all-equal groups carry no gradient | filter them, sample more | DAPO | arithmetic |
| cut-offs | a cut-off completion may be scored unfairly | mask it, or penalize gently | DAPO | design choice |
| KL term | may limit a long-reasoning model | drop it | DAPO, Dr. GRPO | design choice |
| ratio | per-token ratio, per-completion reward | one ratio per completion | GSPO | paper's argument |

> **Options, not a ranking: each paper tested its own setting.**


## Change 1 of 7: how the loss is averaged

**Why this section?** The averaging decides how much each token counts.

> **GRPO divides by the completion's own length first, so each token of a long completion counts less than each token of a short one.**

<img src="images/rllm6_weights.png" width=62%>

- a group of $G = 2$ completions, of $10$ and $100$ tokens
    - per-token share 0.0500 against 0.0050: 10 : 1 (computed here)
- **token-level averaging** (DAPO): divide by all tokens in the group, 0.0091 each
- **Dr. GRPO:** divide by a constant

**Why the papers say it matters**

- **DAPO** (its argument): tokens of long completions contribute less, so good long reasoning is learned slowly and long gibberish is penalized little
- **Dr. GRPO** (its argument): the length factor $1/\lvert \y_i \rvert$ multiplies the advantage, so it also changes how hard a wrong completion is punished
    - take advantage $-1$ and ignore the group factor: a 20-token wrong completion has share $0.05$ per token
    - a 200-token wrong one has $0.005$
    - so wrong completions are punished less per token the longer they are, and the policy can drift toward long wrong ones
- the DAPO ablation: this change added the least accuracy but made training more stable (paper's words)


## Change 2 of 7: the advantage scale

**Why this section?** Dividing by the group's standard deviation changes how much each question counts.

> **Dr. GRPO argues that dividing by the group standard deviation makes almost-all-wrong and almost-all-right questions count more than evenly split ones.**

<img src="images/rllm6_advscale.png" width=62%>

- group of $8$, rewards in $\{0, 1\}$: a lone correct completion gets advantage 2.47 with the division and 0.88 without
- a $4$-of-$8$ question gives 0.94 against 0.50: the lone success is amplified 2.6 times with the division, 1.8 times without


## Change 2, continued: why that is a problem, and who keeps it

**Why this section?** The amplification is arithmetic; whether it is harmful is the paper's argument.

> **Dr. GRPO calls the amplification a difficulty bias (our paraphrase): it comes from the scale of the rewards, not from the question teaching more.**

- **our reading of their argument:** both questions are equally informative
    - yet with the division a lone success counts 2.6 times a 4-of-8 success, and without it 1.8 times
    - so the division alone adds a factor of 1.5
- Dr. GRPO therefore uses reward minus group mean, with no division
- DAPO and GSPO keep the division, so dropping it is an open choice
- common PPO practice normalizes over the whole batch; this normalization is per question


## Change 3 of 7: clip higher

**Why this section?** The clip of Intro session B4 is symmetric; the DAPO paper argues that is the wrong shape for exploration.

> **A ratio clip of $1 + \epsilon$ is the same relative cap for every token, so a likely token can gain a lot of probability and a rare token very little.**

<img src="images/rllm6_clip.png" width=58%>

- with $\epsilon = 0.2$ and a positive advantage, the new probability is at most $1.2 \times$ old
    - $0.01 \to 0.012$; $0.9 \to$ capped at $1$
- **entropy collapse** (observed by the paper): the spread of the token distribution shrinks until a prompt's completions are nearly identical

**The fix: clip higher**

- **the fix:** clip to $[\,1 - \epsilon_{\text{low}},\; 1 + \epsilon_{\text{high}}\,]$
    - the paper's setting: $\epsilon_{\text{low}} = 0.2$, $\epsilon_{\text{high}} = 0.28$
    - the paper keeps $\epsilon_{\text{low}}$ low: raising it would push low-probability tokens toward zero
- the gain is modest: the largest rise of a token at $0.01$ goes from $0.002$ to $0.0028$ (see the figure)
- in the figure the largest rise peaks near $0.8$ and falls: a probability cannot pass $1$
- **risk:** a wider range allows bigger steps, and it was tuned in one setting


## Change 4 of 7: do not waste groups

**Why this section?** Some prompts teach nothing at the current policy.

> **If every completion in a group earns the same reward, all advantages are zero and the group contributes no gradient; DAPO filters such groups out and samples more.**

<img src="images/rllm6_dynamic.png" width=58%>

- a prompt answered correctly with probability $q$ gives an all-equal group of $8$ with probability $q^8 + (1-q)^8$: 0.008 at $q=0.5$, 0.43 at $q=0.9$
- as the policy improves, more prompts move toward the right-hand end (our reading)
- the figure's group of $16$ wastes fewer prompts than a group of $8$
- **dynamic sampling:** keep a prompt only if its group has a success and a failure
    - sample until the batch is full
    - the paper notes more generation per step, but fewer steps to the same accuracy


## Change 5 of 7: do not punish cut-offs unfairly

**Why this section?** Generation stops at a length limit; the completion is then cut off.

> **A cut-off completion may get a punishing reward even if its reasoning was sound; the DAPO paper says this adds noise.**

- **overlong filtering:** mask the loss of cut-off completions
- **soft overlong punishment:** a penalty that grows only near the limit, added to the correctness reward
    - the paper's setting: no penalty up to $16\,384$ tokens (the expected maximum length)
    - then a linear penalty from $0$ down to $-1$ over the next $4\,096$ tokens, so generation stops at $20\,480$
- both are **design choices** about how to score a cut-off
- the paper's evidence is the ablation, not a proof


## Change 6 of 7: drop the KL term

**Why this section?** In Intro session C1 and RLLM-5 the KL kept the policy near a trusted model; two papers remove it, with different reasons.

> **Both papers set the KL coefficient to zero; they give different reasons, so the choice depends on the reward.**

- **DAPO:** a long-reasoning model should move far from its start, so the restriction is unnecessary
- **Dr. GRPO:** with a verifier (a program that checks the answer) there is no learned reward model to exploit, so no anchor is needed
    - it assumes $\beta = 0$ throughout

**What follows, and whose reading it is**

- with a learned reward model the KL keeps the policy where that model is accurate (RLLM-2)
- with no KL, only the reward and the clip hold the policy
- **our reading, not a claim of either paper:** the two arguments together say "drop it with a verifier, think twice with a learned reward model"
- the benefit: no reference model in memory
- the risk: drift that the reward does not forbid


## Change 7 of 7: a ratio for the whole completion

**Why this section?** The reward is one number per completion, but the ratio in the objective is per token.

> **GSPO defines the importance ratio, and its clip, on the whole completion instead of on each token.**

<img src="images/rllm6_ratio.png" width=56%>

- the **completion ratio** is the geometric mean of the token ratios: $s_i(\theta) = \exp\bigl(\frac{1}{\lvert \y_i \rvert} \sum_{\tt} \log r_{i,\tt}(\theta)\bigr)$ (the paper's "sequence ratio")
- the figure is **synthetic**: made-up token ratios show how one noisy token can be clipped while the completion as a whole barely moved
    - its clip range $[0.8, 1.2]$ is drawn for visibility
    - GSPO's real ranges are far narrower (next slide)
- the paper argues token ratios make the gradient noisy and unstable
    - the picture shows the mechanism, not a measurement

**The details, and what to hold back**

- GSPO's clip ranges are $3 \times 10^{-4}$ (below) and $4 \times 10^{-4}$ (above), far narrower than $0.2$
    - not a typo: the completion ratio is a mean over tokens
    - over $1000$ tokens, a mean log ratio of $4 \times 10^{-4}$ is a total of $0.4$ (our arithmetic)
- the paper reports more stable training, and a benefit for **mixture-of-experts (MoE)** models
    - an MoE model routes each token to a few expert sub-networks
    - after an update the routing can change, so token ratios swing
    - **routing replay**, the paper's workaround: reuse the old router choices; GSPO makes it unnecessary
- the evidence is from the paper's own Qwen models


## What the refinements leave open

**Why this section?** They fix GRPO's loss and sampling. They do not change what the signal is.

> **All the recipes keep one number per completion, from samples of the current policy. The credit problem and the sample problem remain.**

- **credit:** every token of a completion still gets the same advantage (the contrast with PPO in RLLM-5)
- **samples:** each batch comes from the current policy, and generating it is paid for again at every step
- **signal:** one scalar reward per completion
    - unused: a stronger model's token probabilities, a critique
- RLLM-7 to RLLM-13 ask three questions of any such recipe (named in RLLM-7)
    - whose samples does it learn from?
    - what does the update consume?
    - how fine is the signal?


# What we established, and what it sets up

- **Established**
    - plain GRPO fell short of a reported result
    - in the DAPO ablation, five steps each helped
    - the objective has seven changeable parts
    - only two (averaging, empty groups) are plain arithmetic; the rest are arguments or design choices
    - **DAPO:** token-level averaging, a higher upper clip, dynamic sampling, overlong handling, no KL
    - **Dr. GRPO:** no division by length or by the standard deviation, and no KL term
    - **GSPO:** a ratio and a clip on the whole completion
    - the evidence is from each paper's own setting: options, not a ranking
- **Sets up**
    - **the limitation:** one scalar per completion, from the current policy's own samples
    - **RLLM-7, the start of part 2 (RFT Alternative methods):** other ways to do RFT
    - they change whose samples are used, what the update consumes, and how fine the signal is
- **Papers:** DAPO 2503.14476; Dr. GRPO ("Understanding R1-Zero-Like Training") 2503.20783; GSPO 2507.18071 (arXiv)
